# Lesson 02: Introduction to AI Agents & `TravelAgent` with Smoke Tests

In this notebook, you will:
1. Implement a `TravelAgent` using `google-genai` (`gemini-2.5-flash`) with a Python `get_random_destination` tool.
2. Compare Simple Reflex, Goal-Based, and Utility-Based agent decision loops.
3. Run an automated smoke-test assertion check on the generated itinerary.

In [ ]:
!pip install -q -U google-genai python-dotenv

In [ ]:
import os, random
from google import genai
from google.genai import types

def get_gemini_client():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    api_key = api_key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=api_key) if api_key else None

def get_random_destination() -> str:
    """Provides a random vacation destination."""
    destinations = ["Barcelona, Spain", "Paris, France", "Tokyo, Japan", "Kyoto, Japan", "Lisbon, Portugal"]
    return random.choice(destinations)

client = get_gemini_client()

In [ ]:
prompt = "Plan me a day trip to a random destination."
if client:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are a helpful AI Travel Agent. Always call get_random_destination first.",
            tools=[get_random_destination]
        )
    )
    answer = response.text
else:
    dest = get_random_destination()
    answer = f"[Offline Simulation] Day trip to {dest}: Morning walking tour, local lunch, afternoon museum visit."

print(answer)

In [ ]:
# Smoke-Test Assertion Check
assertions = {"contains_any": ["day trip", "trip", "itinerary"], "contains_none": ["traceback", "exception"]}
ok = any(t in answer.lower() for t in assertions["contains_any"]) and not any(b in answer.lower() for b in assertions["contains_none"])
print("Smoke test passed:", ok)